### Client Reactivation Business Case — Solution Notebook

This notebook addresses the business case to analyze client inactivity and propose a reactivation strategy.

- Datasets located at: `C:\Users\piers\Desktop\home\Sviluppo\text-llm-learning\fendi_bc\Data Scientist Business Case\Data Scientist Business Case`
- Inactivity rule: A client is inactive after 120 days since last purchase

Business questions:
1. What can you tell about the client population?
2. Can we estimate the probability that an inactive client will return?
3. How do different communication channels (email, call, SMS, letter) influence reactivation?
4. How should we segment and prioritize inactive clients for targeted campaigns?

This notebook follows a structured data science approach: understanding data, feature engineering, modeling, and actionable insights.


In [1]:
# Standard imports
import os
import sys
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')

print('Libraries loaded.')


Libraries loaded.


In [2]:
# Paths
DATA_DIR = r"C:\\Users\\piers\\Desktop\\home\\Sviluppo\\text-llm-learning\\fendi_bc\\Data Scientist Business Case\\Data Scientist Business Case"
CUSTOMERS_PATH = os.path.join(DATA_DIR, 'customers.csv')
TRANSACTIONS_PATH = os.path.join(DATA_DIR, 'transactions.csv')
CAMPAIGNS_PATH = os.path.join(DATA_DIR, 'campaign_tasks.csv')

for p in [CUSTOMERS_PATH, TRANSACTIONS_PATH, CAMPAIGNS_PATH]:
    print(p, 'exists:', os.path.exists(p))


C:\\Users\\piers\\Desktop\\home\\Sviluppo\\text-llm-learning\\fendi_bc\\Data Scientist Business Case\\Data Scientist Business Case\customers.csv exists: True
C:\\Users\\piers\\Desktop\\home\\Sviluppo\\text-llm-learning\\fendi_bc\\Data Scientist Business Case\\Data Scientist Business Case\transactions.csv exists: True
C:\\Users\\piers\\Desktop\\home\\Sviluppo\\text-llm-learning\\fendi_bc\\Data Scientist Business Case\\Data Scientist Business Case\campaign_tasks.csv exists: True


In [3]:
# Utility functions

def parse_dates(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Convert dates columns into pd.to_datetime columns if columns name in cols."""
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors='coerce', utc=True).dt.tz_localize(None)
    return df


def summarize_dataframe(df: pd.DataFrame, name: str) -> None:
    """Just return extra information about dataset (shape, % Null columns, ...)"""
    print(f"\n=== {name} ===")
    print('Shape:', df.shape)
    print('Columns:', list(df.columns))
    print('Null % by column:')
    print((df.isna().mean() * 100).round(2).sort_values(ascending=False).head(20))


def check_required_columns(df: pd.DataFrame, required: list[str], name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {name}: {missing}")

INACTIVITY_DAYS = 120
INACTIVITY_DELTA = timedelta(days=INACTIVITY_DAYS)



In [4]:
# Load datasets
customers = pd.read_csv(CUSTOMERS_PATH)
transactions = pd.read_csv(TRANSACTIONS_PATH)
campaigns = pd.read_csv(CAMPAIGNS_PATH)

# Basic normalization
customers.columns = [c.strip().lower() for c in customers.columns]
transactions.columns = [c.strip().lower() for c in transactions.columns]
campaigns.columns = [c.strip().lower() for c in campaigns.columns]

# # Parse dates cols to datetime:
transactions = parse_dates(transactions, ['event_date'])
campaigns = parse_dates(campaigns, ['start_date', 'end_date', 'last_modified_date'])
customers = parse_dates(customers, ['registration_date'])

summarize_dataframe(customers, 'customers')
summarize_dataframe(transactions, 'transactions')
summarize_dataframe(campaigns, 'campaigns')



=== customers ===
Shape: (3500, 7)
Columns: ['customer_id', 'birth_year', 'gender', 'country', 'registration_date', 'is_active', 'marketing_consent']
Null % by column:
customer_id          0.0
birth_year           0.0
gender               0.0
country              0.0
registration_date    0.0
is_active            0.0
marketing_consent    0.0
dtype: float64

=== transactions ===
Shape: (7000, 6)
Columns: ['customer_id', 'event_type', 'event_date', 'product_category', 'quantity', 'net_sale_euro']
Null % by column:
customer_id         0.0
event_type          0.0
event_date          0.0
product_category    0.0
quantity            0.0
net_sale_euro       0.0
dtype: float64

=== campaigns ===
Shape: (5000, 8)
Columns: ['id_campaign', 'customer_id', 'campaign_status', 'campaign_channel', 'activity_subject', 'start_date', 'end_date', 'last_modified_date']
Null % by column:
id_campaign           0.0
customer_id           0.0
campaign_status       0.0
campaign_channel      0.0
activity_subject  

## 0.0.0 Reactivated customers:

In [8]:
def reactivated_client(df_trans, id_col, date_col, daydiff):
    sample = df_trans.sort_values([id_col, date_col])
    
    # shift last buy:
    sample['date_last_purchase'] = sample.groupby([id_col])[date_col].shift(1)
    sample['date_last_purchase'] = sample['date_last_purchase'].combine_first(sample[date_col])
    sample['recency_days'] = (sample[date_col] - sample['date_last_purchase']).dt.days
    sample['is_reactived'] = sample['recency_days'] >= daydiff 

    # sample = sample[sample['recency_days'] >= daydiff]

    return sample

In [9]:
reactivated_client(
    df_trans=transactions, 
    id_col='customer_id', 
    date_col='event_date', 
    daydiff=INACTIVITY_DAYS
)[['customer_id', 'product_category', 'quantity', 'net_sale_euro', 'date_last_purchase', 'event_date', 'recency_days', 'is_reactived']].sort_values('customer_id')

,customer_id,product_category,quantity,net_sale_euro,date_last_purchase,event_date,recency_days,is_reactived
5367,CUST_00001,slg,3,1209.93,2024-08-17,2024-08-17,0,False
539,CUST_00001,bags,1,1784.23,2024-08-17,2024-12-16,121,True
5365,CUST_00002,textile,2,1392.48,2024-06-08,2024-06-08,0,False
6956,CUST_00002,shoes,2,1328.70,2024-06-08,2024-08-27,80,False
6418,CUST_00002,bags,1,1417.73,2024-08-27,2024-09-06,10,False
...,...,...,...,...,...,...,...,...
5480,CUST_03497,bags,1,1967.04,2024-07-12,2024-07-12,0,False
5159,CUST_03497,slg,2,154.79,2024-07-12,2024-11-10,121,True
321,CUST_03498,bags,4,794.72,2024-08-22,2024-11-30,100,False
2447,CUST_03498,slg,2,1779.45,2024-08-22,2024-08-22,0,False


In [14]:
customers['marketing_consent'].value_counts()

marketing_consent
1    3149
0     351
Name: count, dtype: int64

In [113]:
campaigns[campaigns['customer_id'].isin(['CUST_00002', 'CUST_00003', 'CUST_00018'])].sort_values(['customer_id', 'start_date'])

,id_campaign,customer_id,campaign_status,campaign_channel,activity_subject,start_date,end_date,last_modified_date
4381,CAMPAIGN_04382,CUST_00002,Done,email,Loyalty Program Update,2023-08-15,2023-08-20,2023-08-25
4117,CAMPAIGN_04118,CUST_00003,Done,email,Holiday Sale,2023-04-27,2023-04-30,2023-05-04
4703,CAMPAIGN_04704,CUST_00003,Done,letter,Exclusive Preview,2024-01-07,2024-01-10,2024-01-15
3967,CAMPAIGN_03968,CUST_00003,Done,sms,Loyalty Program Update,2024-09-15,2024-09-24,2024-09-29
1128,CAMPAIGN_01129,CUST_00018,Done,call,Exclusive Preview,2024-03-23,2024-04-01,2024-04-06
1290,CAMPAIGN_01291,CUST_00018,Done,email,Holiday Sale,2024-12-21,2024-12-29,2024-12-31


In [23]:
# Standardize expected column names (best-effort)

# Guess id columns
cust_id_col = None
for c in ['customer_id']:
    if c in customers.columns:
        cust_id_col = c
        break

trx_date_col = None
for c in ['event_date']:
    if c in transactions.columns and np.issubdtype(transactions[c].dtype, np.datetime64):
        trx_date_col = c
        break

trx_amount_col = None
for c in ['net_sale_euro']:
    if c in transactions.columns:
        trx_amount_col = c
        break


camp_date_start_col = None
for c in ['start_date']:
    if c in campaigns.columns and np.issubdtype(campaigns[c].dtype, np.datetime64):
        camp_date_start_col = c
        break

camp_date_end_col = None
for c in ['end_date']:
    if c in campaigns.columns and np.issubdtype(campaigns[c].dtype, np.datetime64):
        camp_date_end_col = c
        break

camp_channel_col = None
for c in ['campaign_channel']:
    if c in campaigns.columns:
        camp_channel_col = c
        break

if cust_id_col is None:
    raise ValueError('Could not infer customer id column in customers.csv')

# Ensure customers has id column name `customer_id`
if cust_id_col != 'customer_id':
    customers = customers.rename(columns={cust_id_col: 'customer_id'})
    if 'customer_id' in transactions.columns:
        pass
    else:
        # Try to map in transactions
        for c in ['customer_id', 'client_id', 'id', 'cust_id']:
            if c in transactions.columns:
                transactions = transactions.rename(columns={c: 'customer_id'})
                break
    if 'customer_id' in campaigns.columns:
        pass
    else:
        for c in ['customer_id', 'client_id', 'id', 'cust_id']:
            if c in campaigns.columns:
                campaigns = campaigns.rename(columns={c: 'customer_id'})
                break

required_trx = ['customer_id']
if trx_date_col is None:
    raise ValueError('Could not infer a transaction date column in transactions.csv')
required_trx.append(trx_date_col)

if trx_amount_col is None:
    print('Warning: amount column not found, defaulting to 1 per transaction for frequency-only metrics')

required_camp = ['customer_id']
if camp_date_start_col is None:
    print('Warning: campaign date column not found; channel impact will be limited')
else:
    required_camp.append(camp_date_start_col)
if camp_channel_col is None:
    print('Warning: campaign channel column not found; will treat as unknown')

required_camp = ['customer_id']
if camp_date_end_col is None:
    print('Warning: campaign date column not found; channel impact will be limited')
else:
    required_camp.append(camp_date_end_col)
if camp_channel_col is None:
    print('Warning: campaign channel column not found; will treat as unknown')

check_required_columns(transactions, required_trx, 'transactions')
check_required_columns(campaigns, required_camp, 'campaigns')
print(f"""Standardization complete:
    - 'trx_date_col': {trx_date_col},
    - 'trx_amount_col': {trx_amount_col},
    - 'camp_date_start_col': {camp_date_start_col},
    - 'camp_date_start_col': {camp_date_end_col},
    - 'camp_channel_col': {camp_channel_col},
""")


Standardization complete:
    - 'trx_date_col': event_date,
    - 'trx_amount_col': net_sale_euro,
    - 'camp_date_start_col': start_date,
    - 'camp_date_start_col': end_date,
    - 'camp_channel_col': campaign_channel,



In [44]:
# Inactivity features (RFM-like)

# Compute per-customer last purchase date, frequency, and monetary
trx = transactions.dropna(subset=['customer_id', trx_date_col]).copy()
if trx_amount_col is None:
    trx['amount_proxy'] = 1.0
    amount_col = 'amount_proxy'
else:
    amount_col = trx_amount_col

# Dataframe rappresent the last purchase per-customer:
cust_agg = trx.groupby('customer_id').agg(
    last_purchase_date=(trx_date_col, 'max'), # last date
    purchase_frequency=('customer_id', 'count'), # number of purchase
    monetary_value=(amount_col, 'sum'), # total money spent
).reset_index()

# Define analysis reference date as max available date inside transactions dataframe. 
# If there is not max available date, we use today...
reference_date = trx[trx_date_col].max() # find max data
if pd.isna(reference_date): # if not
    reference_date = pd.Timestamp.today().normalize() # today

cust_agg['recency_days'] = (reference_date - cust_agg['last_purchase_date']).dt.days
cust_agg['is_inactive'] = cust_agg['recency_days'] >= INACTIVITY_DAYS # 120 days

# Merge with customers
customers_feat = customers.merge(cust_agg, on='customer_id', how='left')
customers_feat['purchase_frequency'] = customers_feat['purchase_frequency'].fillna(0)
customers_feat['monetary_value'] = customers_feat['monetary_value'].fillna(0.0)
customers_feat['recency_days'] = customers_feat['recency_days'].fillna(np.inf)
customers_feat['is_inactive'] = customers_feat['is_inactive'].fillna(True)

customers_feat.head(3)

C:\Users\piers\AppData\Local\Temp\ipykernel_14848\44453125.py:32: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  customers_feat['is_inactive'] = customers_feat['is_inactive'].fillna(True)


,customer_id,birth_year,gender,country,registration_date,is_active,marketing_consent,last_purchase_date,purchase_frequency,monetary_value,recency_days,is_inactive
0,CUST_00001,1988,Other,CN,2024-08-02,1,1,2024-12-16,2.0,2994.16,131.0,True
1,CUST_00002,2001,M,US,2021-08-17,1,1,2024-09-06,3.0,4138.91,232.0,True
2,CUST_00003,1978,M,CN,2021-02-21,1,1,2024-12-12,2.0,2773.93,135.0,True


In [47]:
cust_agg.groupby(['is_inactive'])['customer_id'].count()

is_inactive
False     363
True     2657
Name: customer_id, dtype: int64

In [46]:
customers_feat.groupby(['is_active', 'is_inactive'])['customer_id'].count()

is_active  is_inactive
0          True           1112
1          False           363
           True           2025
Name: customer_id, dtype: int64

### Q1. Client population overview (EDA)


In [ ]:
# Basic EDA visuals/tables
print('Total customers:', customers_feat['customer_id'].nunique())
print('Inactive share (%):', (customers_feat['is_inactive'].mean() * 100).round(2))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(customers_feat['recency_days'].replace(np.inf, np.nan).dropna(), bins=30, ax=axes[0])
axes[0].set_title('Recency (days)')

sns.histplot(customers_feat['purchase_frequency'], bins=30, ax=axes[1])
axes[1].set_title('Purchase Frequency')

sns.histplot(customers_feat['monetary_value'], bins=30, ax=axes[2])
axes[2].set_title('Monetary Value')

plt.tight_layout()
plt.show()

customers_feat[['is_inactive','purchase_frequency','monetary_value']].describe(include='all')


### Q2. Estimate probability that an inactive client will return


In [ ]:
# Label definition: return within horizon
# Define a horizon (e.g., next 60 days after reference_date)
LABEL_HORIZON_DAYS = 60
horizon_end = reference_date + pd.Timedelta(days=LABEL_HORIZON_DAYS)

# For each inactive customer at reference_date, did they purchase within horizon?
future_trx = trx[(trx[trx_date_col] > reference_date) & (trx[trx_date_col] <= horizon_end)]
returned_customers = set(future_trx['customer_id'].unique())

model_df = customers_feat.copy()
model_df = model_df[model_df['is_inactive'] == True].copy()
model_df['returned_within_horizon'] = model_df['customer_id'].isin(returned_customers).astype(int)

# Simple feature set
feature_cols = ['recency_days', 'purchase_frequency', 'monetary_value']
model_df = model_df.replace({np.inf: np.nan}).fillna(0)

print('Inactive samples:', len(model_df))
print('Positive rate:', model_df['returned_within_horizon'].mean().round(4))
model_df[feature_cols + ['returned_within_horizon']].head()


In [ ]:
# Baseline model (Logistic Regression)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

X = model_df[feature_cols]
y = model_df['returned_within_horizon']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

clf.fit(X_train, y_train)
proba = clf.predict_proba(X_test)[:, 1]
print('ROC AUC:', roc_auc_score(y_test, proba).round(4))
print('PR AUC:', average_precision_score(y_test, proba).round(4))


### Q3. Channel impact on reactivation


In [ ]:
# Build simple channel -> reactivation rate analysis

if camp_date_col is not None:
    # Consider contacts in the horizon window and next purchase
    camp = campaigns.dropna(subset=['customer_id', camp_date_col]).copy()
    camp = camp[camp[camp_date_col] <= horizon_end]
    if camp_channel_col is None:
        camp['channel_eval'] = 'unknown'
    else:
        camp['channel_eval'] = camp[camp_channel_col].fillna('unknown').astype(str).str.lower()

    # For each customer, if contacted via channel X during the window, did they return?
    contact_flags = camp.groupby(['customer_id', 'channel_eval']).size().unstack(fill_value=0).astype(bool).astype(int)

    # Attach label of returned_within_horizon
    inactive_labels = model_df.set_index('customer_id')['returned_within_horizon']
    channel_df = contact_flags.join(inactive_labels, how='inner')

    # Reactivation rate by channel exposure (1 vs 0)
    channel_impact = {}
    for ch in [c for c in channel_df.columns if c != 'returned_within_horizon']:
        exposed = channel_df[channel_df[ch] == 1]['returned_within_horizon']
        not_exposed = channel_df[channel_df[ch] == 0]['returned_within_horizon']
        channel_impact[ch] = {
            'n_exposed': int(len(exposed)),
            'rate_exposed': float(exposed.mean()) if len(exposed) else np.nan,
            'n_not_exposed': int(len(not_exposed)),
            'rate_not_exposed': float(not_exposed.mean()) if len(not_exposed) else np.nan,
            'lift_pp': float((exposed.mean() - not_exposed.mean()) * 100) if len(exposed) and len(not_exposed) else np.nan,
        }
    impact_table = pd.DataFrame(channel_impact).T.sort_values('lift_pp', ascending=False)
    display(impact_table)
else:
    print('Campaign date column not available; skipping channel impact analysis.')


### Q4. Segmentation and prioritization for campaigns


In [ ]:
# Scoring and simple segmentation

# Score every inactive customer with model probabilities (train on all for scoring)
full_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
full_clf.fit(model_df[feature_cols], model_df['returned_within_horizon'])

customers_scored = model_df[['customer_id'] + feature_cols].copy()
customers_scored['pred_return_proba'] = full_clf.predict_proba(customers_scored[feature_cols])[:, 1]

# Monetary potential proxy
customers_scored = customers_scored.merge(customers_feat[['customer_id', 'monetary_value']], on='customer_id', how='left')

# Segments by probability and value
customers_scored['proba_band'] = pd.qcut(customers_scored['pred_return_proba'].rank(method='first'), 4, labels=['P1 (Low)', 'P2', 'P3', 'P4 (High)'])
customers_scored['value_band'] = pd.qcut(customers_scored['monetary_value'].rank(method='first'), 4, labels=['V1 (Low)', 'V2', 'V3', 'V4 (High)'])

# Priority rule: target P4 & V3/V4 first, then P3 & V3/V4, etc.
def assign_priority(row):
    p = str(row['proba_band'])
    v = str(row['value_band'])
    if p.startswith('P4') and v in ['V4 (High)', 'V3']:
        return 'Tier 1'
    if p.startswith('P3') and v in ['V4 (High)', 'V3']:
        return 'Tier 2'
    if p.startswith('P4'):
        return 'Tier 2'
    if p.startswith('P2') and v in ['V4 (High)', 'V3']:
        return 'Tier 3'
    return 'Tier 4'

customers_scored['priority_tier'] = customers_scored.apply(assign_priority, axis=1)

summary = customers_scored.groupby(['priority_tier']).agg(
    customers=('customer_id', 'nunique'),
    avg_proba=('pred_return_proba', 'mean'),
    avg_value=('monetary_value', 'mean'),
).sort_values(['priority_tier'])

summary
